# Capa Gold — Modelo analítico, indicadores y vistas de negocio
**Proyecto:** Detección de Fraude en Transacciones con Tarjeta de Crédito
**Objetivo de la capa:** A partir de las tablas Silver (dimensiones + hechos), construir el modelo analítico tipo estrella, vistas SQL consolidadas y tablas agregadas con indicadores (KPIs) de fraude, listas para el consumo del dashboard en Databricks/Power BI y la conciliación en Excel.

In [0]:
catalog = "fraud_proyecto"
schema_silver = "silver"
schema_gold = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_gold}")

fact_transacciones = spark.table(f"{catalog}.{schema_silver}.fact_transacciones")
dim_titulares = spark.table(f"{catalog}.{schema_silver}.dim_titulares_tarjeta")
dim_comercios = spark.table(f"{catalog}.{schema_silver}.dim_comercios")
dim_fechas = spark.table(f"{catalog}.{schema_silver}.dim_fechas")

print(f"fact_transacciones: {fact_transacciones.count():,} registros")

## 1. Vista de modelo estrella consolidada

Vista SQL que une la tabla de hechos con sus tres dimensiones, lista para ser consumida directamente por el dashboard sin requerir joins adicionales en la herramienta de BI.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{schema_gold}.vw_modelo_estrella_transacciones AS
SELECT
    f.id_transaccion,
    f.fecha,
    f.timestamp_transaccion,
    f.hora,
    f.dia_de_la_semana,
    f.es_madrugada,
    f.es_hora_pico,
    f.edad_del_titular,
    f.monto,
    f.es_fraude,
    f.latitud_comercio,
    f.longitud_comercio,
    t.id_titular,
    t.genero_titular,
    t.ciudad_titular,
    t.estado_titular,
    t.poblacion_ciudad,
    t.ocupacion_titular,
    c.id_comercio,
    c.nombre_comercio,
    f.nombre_categoria,
    d.anio,
    d.mes,
    d.dia,
    d.nombre_dia
FROM {catalog}.{schema_silver}.fact_transacciones f
LEFT JOIN {catalog}.{schema_silver}.dim_titulares_tarjeta t ON f.id_titular = t.id_titular
LEFT JOIN {catalog}.{schema_silver}.dim_comercios c ON f.nombre_comercio = c.nombre_comercio
LEFT JOIN {catalog}.{schema_silver}.dim_fechas d ON f.fecha = d.fecha
""")

display(spark.table(f"{catalog}.{schema_gold}.vw_modelo_estrella_transacciones"))

## 2. Indicador (KPI): tasa de fraude por categoría de comercio

In [0]:
kpi_categoria = spark.sql(f"""
SELECT
    nombre_categoria,
    COUNT(*) AS total_transacciones,
    ROUND(SUM(monto), 2) AS monto_total,
    ROUND(AVG(monto), 2) AS monto_promedio,
    SUM(es_fraude) AS total_fraudes,
    ROUND(100.0 * SUM(es_fraude) / COUNT(*), 2) AS tasa_fraude_pct
FROM {catalog}.{schema_silver}.fact_transacciones
GROUP BY nombre_categoria
ORDER BY tasa_fraude_pct DESC
""")

(
    kpi_categoria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_gold}.kpi_fraude_por_categoria")
)

display(kpi_categoria)

## 3. Indicador (KPI): tasa de fraude por franja horaria

**¿Cómo ayuda al negocio?** Permite priorizar reglas de monitoreo y revisión manual en las franjas horarias (madrugada / hora pico) donde se concentra proporcionalmente más fraude.

In [0]:
kpi_horario = spark.sql(f"""
SELECT
    hora,
    es_madrugada,
    es_hora_pico,
    COUNT(*) AS total_transacciones,
    SUM(es_fraude) AS total_fraudes,
    ROUND(100.0 * SUM(es_fraude) / COUNT(*), 2) AS tasa_fraude_pct
FROM {catalog}.{schema_silver}.fact_transacciones
GROUP BY hora, es_madrugada, es_hora_pico
ORDER BY hora
""")

(
    kpi_horario.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_gold}.kpi_fraude_por_horario")
)

display(kpi_horario)

## 4. Indicador (KPI): tasa de fraude por estado del titular

**¿Cómo ayuda al negocio?** Permite identificar zonas geográficas con mayor incidencia relativa de fraude para focalizar campañas de prevención o alertas regionales.

In [0]:
kpi_estado = spark.sql(f"""
SELECT
    t.estado_titular,
    COUNT(*) AS total_transacciones,
    SUM(f.es_fraude) AS total_fraudes,
    ROUND(100.0 * SUM(f.es_fraude) / COUNT(*), 2) AS tasa_fraude_pct
FROM {catalog}.{schema_silver}.fact_transacciones f
LEFT JOIN {catalog}.{schema_silver}.dim_titulares_tarjeta t ON f.id_titular = t.id_titular
GROUP BY t.estado_titular
ORDER BY tasa_fraude_pct DESC
""")

(
    kpi_estado.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_gold}.kpi_fraude_por_estado")
)

display(kpi_estado)

## 5. Ranking de comercios con mayor riesgo de fraude

**¿Cómo ayuda al negocio?** Identifica comercios específicos que deberían someterse a mayor escrutinio o validaciones adicionales antes de aprobar una transacción. Se exige un mínimo de transacciones para evitar sesgos por bajo volumen.

In [0]:
MIN_TRANSACCIONES = 5

ranking_comercios = spark.sql(f"""
SELECT
    nombre_comercio,
    COUNT(*) AS total_transacciones,
    SUM(es_fraude) AS total_fraudes,
    ROUND(100.0 * SUM(es_fraude) / COUNT(*), 2) AS tasa_fraude_pct
FROM {catalog}.{schema_silver}.fact_transacciones
GROUP BY nombre_comercio
HAVING COUNT(*) >= {MIN_TRANSACCIONES}
ORDER BY tasa_fraude_pct DESC, total_transacciones DESC
""")

(
    ranking_comercios.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_gold}.ranking_comercios_riesgo")
)

display(ranking_comercios.limit(20))

## 6. Resumen diario de transacciones (para dashboard y conciliación en Excel)

**¿Cómo ayuda al negocio?** Provee la serie de control diaria (volumen, monto y fraude) que se utiliza como base de la conciliación contra los totales calculados de forma independiente en Excel.

In [0]:
kpi_resumen_diario = spark.sql(f"""
SELECT
    fecha,
    COUNT(*) AS total_transacciones,
    ROUND(SUM(monto), 2) AS monto_total,
    SUM(es_fraude) AS total_fraudes,
    ROUND(100.0 * SUM(es_fraude) / COUNT(*), 2) AS tasa_fraude_pct
FROM {catalog}.{schema_silver}.fact_transacciones
GROUP BY fecha
ORDER BY fecha
""")

(
    kpi_resumen_diario.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_gold}.kpi_resumen_diario")
)

display(kpi_resumen_diario)

In [0]:
kpi_resumen_diario_top = spark.sql(f"""
SELECT
    fecha,
    COUNT(*) AS total_transacciones,
    ROUND(SUM(monto), 2) AS monto_total,
    SUM(es_fraude) AS total_fraudes,
    ROUND(100.0 * SUM(es_fraude) / COUNT(*), 2) AS tasa_fraude_pct
FROM fraud_proyecto.silver.fact_transacciones
GROUP BY fecha
ORDER BY tasa_fraude_pct DESC, total_fraudes DESC
LIMIT 15
""")
display(kpi_resumen_diario_top)

## 7. Verificación general de la capa Gold

In [0]:
tablas_gold = [
    "kpi_fraude_por_categoria",
    "kpi_fraude_por_horario",
    "kpi_fraude_por_estado",
    "ranking_comercios_riesgo",
    "kpi_resumen_diario",
]

for tabla in tablas_gold:
    total = spark.table(f"{catalog}.{schema_gold}.{tabla}").count()
    print(f"{catalog}.{schema_gold}.{tabla}: {total:,} filas")

print(f"\nVista de modelo estrella disponible en: {catalog}.{schema_gold}.vw_modelo_estrella_transacciones")

## 8. Exportación de KPIs a CSV (insumo de conciliación en Excel)

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_gold}.export_csv")

for tabla in tablas_gold:
    nombre_tabla = f"{catalog}.{schema_gold}.{tabla}"
    ruta_salida = f"/Volumes/{catalog}/{schema_gold}/export_csv/{tabla}"

    df = spark.table(nombre_tabla)
    (
        df.write
        .format("csv")
        .option("header", "true")
        .option("sep", ",")
        .option("encoding", "UTF-8")
        .mode("overwrite")
        .save(ruta_salida)
    )
    print(f"Exportado {tabla} -> {ruta_salida}")